# 2장 1강: 회귀와 분류 모델의 예측 — 실습문제

## 실습 목표

- 금액을 예측하는 회귀와 구매 여부를 예측하는 분류를 구분합니다.
- 분류의 점수, 예측 확률, 최종 클래스가 각각 무엇인지 구분합니다.
- 교안의 함수를 이용해 Sigmoid·Softmax 확률을 계산합니다.
- 임계값 비교와 Argmax로 최종 클래스를 선택합니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용합니다. 외부 데이터 파일은 필요하지 않습니다. 모든 숫자와 계산 방법은 제공된 「2장 1강: 회귀와 분류 모델의 예측」의 본문 예제와 연습문제를 바탕으로 구성했습니다. 주어진 가중치로 출력만 계산하며 모델을 학습시키는 실습은 없습니다.

## 실습 구성과 교안 대응

| 구분 | 내용 | 교안 근거 및 구성 방식 |
|---|---|---|
| 필수 1 / 문제 1-1 | 회귀 예측값과 분류 점수 | 1절 본문의 고객 3명 예제 |
| 필수 1 / 문제 1-2 | Sigmoid와 임계값 | 1절의 점수에 2·3절의 계산 규칙을 연결 |
| 필수 2 / 문제 2-1 | 한 상품의 Softmax와 Argmax | 2·3절 본문의 [2,1,0] 예제 |
| 필수 2 / 문제 2-2 | 고객별 Softmax와 Argmax | 2·3절 본문의 3행 점수 배열 예제 |
| 과제 / 문제 3-1 | 임계값 0.5와 0.7 비교 | 연습문제 3의 소문항 1·2를 나누어 연습 |
| 과제 / 문제 3-2 | 한 상품의 Softmax 계산 복습 | 연습문제 2의 소문항 3에 3절의 Argmax 적용 |

**교안의 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 교안 본문 예제와 연습문제의 일부를 기초 실습 4문제·복습 과제 2문제로 구성했습니다. 새 공식을 유도하거나 여러 오류 유형을 한꺼번에 분류하는 과제는 요구하지 않습니다.

## 진행 안내

1. NumPy를 불러오는 셀부터 위에서 아래로 실행합니다.
2. 표에서 숫자가 무엇을 뜻하는지 확인한 뒤 계산합니다.
3. 함수 코드는 문제에 제공된 교안의 형태를 그대로 사용해도 됩니다. `return` 앞의 들여쓰기를 유지하세요.
4. 확률을 소수점 아래 세 자리로 표시할 때만 `np.round()`를 씁니다. 임계값과 비교할 때는 반올림 전 확률을 사용합니다.
5. 각 문제의 함수와 변수를 다시 작성합니다. 한 벡터용 Softmax와 여러 행용 Softmax는 사용하는 코드가 다릅니다.
6. 해석 질문의 답은 계산값과 함께 쉬운 문장으로 적으면 됩니다.

## 먼저 읽는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 입력 특성 | 계산에 넣는 정보. 여기서는 방문 횟수와 장바구니 추가 횟수 |
| 가중치 w / 편향 b | 입력에 각각 곱할 숫자 / 곱해서 더한 결과에 추가로 더할 숫자 |
| 회귀 | 금액처럼 ‘얼마인가’를 숫자로 예측하는 문제 |
| 분류 / 클래스 | 정해진 종류 중 하나를 고르는 문제 / 그 종류의 이름 |
| 이진 분류 | 두 종류 중 하나를 선택하는 분류. 여기서는 구매·미구매 |
| 다중 클래스 분류 | 세 개 이상의 종류 중 하나를 선택하는 분류. 여기서는 의류·식품·가전 |
| 점수 z / Logit(로짓) | 확률로 바꾸기 전 모델이 계산한 숫자 |
| 확률 p | 모델이 어떤 클래스에 부여한 가능성을 0~1의 형태로 나타낸 값. 0.7은 70% |
| 실제값 y / 예측값 ŷ | 실제로 관측된 정답 / 모델이 계산한 결과. ŷ는 ‘와이 햇’으로 읽음 |
| Sigmoid(시그모이드) | 이진 분류에서 점수 하나를 클래스 1의 확률로 바꾸는 함수 |
| Softmax(소프트맥스) | 한 데이터의 클래스별 점수를 합이 1인 확률들로 바꾸는 함수 |
| Threshold(임계값) | 확률을 0 또는 1로 결정할 때 비교하는 기준값 |
| Argmax(아그맥스) | 가장 큰 값이 있는 위치 번호를 찾는 연산 |
| 인덱스 | 배열의 위치 번호. Python에서는 첫 위치가 0 |
| 행 / 열 / Shape | 가로줄 / 세로줄 / 배열의 모양. (3,2)는 3행 2열 |

이 실습의 이진 분류에서 **0은 미구매, 1은 구매**입니다. 다중 클래스의 인덱스는 **0=의류, 1=식품, 2=가전**입니다. 같은 숫자 1이어도 이진 분류의 클래스 1과 다중 클래스의 인덱스 1은 서로 다른 이름을 가리킵니다.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 숫자 배열을 계산하는 도구이며, `np`는 NumPy를 짧게 부르는 이름입니다.

In [ ]:
import numpy as np

---

## 필수 1. 회귀 예측과 이진 분류 이해하기

### 문제 1-1. 같은 입력으로 회귀 예측값과 분류 점수 계산하기

#### 문제 설명

온라인 쇼핑몰 고객의 활동 데이터입니다. 교안 1절의 예시를 사용합니다.

| 고객 | 방문 횟수 | 장바구니 추가 횟수 | 실제 구매 금액(원) | 실제 구매 여부 |
|---|---:|---:|---:|---:|
| A | 4 | 2 | 12000 | 1 |
| B | 2 | 0 | 0 | 0 |
| C | 6 | 3 | 24000 | 1 |

입력 X에는 방문 횟수와 장바구니 추가 횟수만 순서대로 넣습니다. 행 순서는 A, B, C입니다. 구매 여부의 0은 미구매, 1은 구매를 뜻합니다.

| 예측 대상 | 가중치 | 편향 | 계산식 |
|---|---|---:|---|
| 구매 금액 | w_reg = [1000, 3000] | b_reg = 5000 | y_reg = X @ w_reg + b_reg |
| 구매 여부의 점수 | w_clf = [0.3, 0.8] | b_clf = −2 | z = X @ w_clf + b_clf |

가중치와 편향은 계산 원리를 확인하기 위해 주어진 값입니다. 이 문제에서는 분류 점수까지만 계산합니다.

#### 계산식을 말로 읽기

`X @ w + b`는 ‘고객 한 명의 첫 입력×첫 가중치 + 둘째 입력×둘째 가중치 + 편향’입니다. `@`는 이러한 곱셈과 덧셈을 고객별로 계산합니다. 이번에는 두 입력을 사용하므로 가중치도 두 개입니다.

#### 요구사항

1. 입력 행렬 `X`와 두 모델의 가중치·편향을 만들고, `X.shape`를 출력하세요.
2. 교안의 `@`와 `+`를 사용하여 세 고객의 예측 구매 금액 `y_reg`와 분류 점수 `z`를 계산하세요. 두 결과와 Shape를 출력하세요.
3. 고객 A의 금액 예측 오차를 “예측 구매 금액 − 실제 구매 금액”으로 계산하고 출력하세요.
4. 고객 A의 두 출력값을 수식에 숫자를 대입하여 확인하고, Q1~Q5에 답하세요.

#### 작성 도움말

`X`는 `np.array([[4, 2], [2, 0], [6, 3]])`으로 만듭니다. `y_reg[0]`은 고객 A의 예측 구매 금액입니다. 실제 금액과 실제 구매 여부는 입력 특성에 넣지 않습니다.

#### 해석 질문

**Q1.** 구매 금액 예측과 구매 여부 예측은 각각 어떤 문제 유형인가요?

**Q2.** X, y_reg, z의 Shape는 무엇이며, 결과 배열의 첫 번째 값은 누구의 결과인가요?

**Q3.** 세 고객의 예측 구매 금액은 각각 얼마인가요? 고객 A의 오차는 무엇을 뜻하나요?

**Q4.** 세 고객의 분류 점수는 각각 얼마인가요? A의 점수 0.8을 구매 확률 80%로 읽어도 되나요?

**Q5.** 주어진 코드로 예측값을 계산하면 가중치와 편향도 자동으로 학습되나요?

#### 제출 결과

- 두 모델의 계산 코드, 출력값과 Shape
- 고객 A의 대입 계산 과정과 금액 예측 오차
- Q1~Q5 답변

In [ ]:
# 문제 1-1. 같은 입력으로 회귀 예측값과 분류 점수 계산하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.

#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 구매 금액 예측과 구매 여부 예측은 각각 어떤 문제 유형인가요?**

- 답변: 

**Q2. X, y_reg, z의 Shape는 무엇이며, 결과 배열의 첫 번째 값은 누구의 결과인가요?**

- 답변: 

**Q3. 세 고객의 예측 구매 금액은 각각 얼마인가요? 고객 A의 오차는 무엇을 뜻하나요?**

- 답변: 

**Q4. 세 고객의 분류 점수는 각각 얼마인가요? A의 점수 0.8을 구매 확률 80%로 읽어도 되나요?**

- 답변: 

**Q5. 주어진 코드로 예측값을 계산하면 가중치와 편향도 자동으로 학습되나요?**

- 답변:

### 문제 1-2. Sigmoid 확률과 Threshold로 구매 여부 결정하기

#### 문제 설명

문제 1-1의 고객별 분류 점수를 사용합니다. 계산의 출발점을 분명히 하기 위해 점수를 다시 배열로 만드세요.

| 고객 | 점수 z |
|---|---:|
| A | 0.8 |
| B | −1.4 |
| C | 2.2 |

교안의 공식 `Sigmoid(z) = 1 / (1 + e^(−z))`로 구매 확률 p를 계산합니다. e는 약 2.718인 자연상수이며, `np.exp(-z)`가 e^(−z)를 계산합니다. 미구매 확률은 `1 − p`입니다.

결정 규칙은 **p가 임계값 이상이면 구매(1), 작으면 미구매(0)**입니다. 임계값은 0.5와 0.7을 각각 적용합니다. 추가로 점수가 정확히 0인 경우도 확인합니다.

#### 수식과 기호 읽기

`1 / (1 + e^(−z))`는 **z의 부호를 바꾼다 → e의 그 제곱값을 구한다 → 1을 더한다 → 그 전체로 1을 나눈다**는 순서입니다. 분수의 위 숫자는 분자, 아래 숫자는 분모입니다. 여기서는 분자가 1, 분모가 `1+e^(−z)`입니다.

```python
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
```

함수는 위 코드를 그대로 사용해도 됩니다. z가 −1.4이면 −z는 +1.4입니다. 음수 앞에 다시 마이너스를 붙이면 양수가 됩니다.

#### 요구사항

1. 교안처럼 `def sigmoid(z):`와 `return 1 / (1 + np.exp(-z))`로 함수를 작성하세요.
2. 점수 배열 `[0.8, -1.4, 2.2]`에 Sigmoid를 적용하여 구매 확률 `p`와 미구매 확률 `1 - p`를 출력하세요.
3. `(p >= 임계값).astype(int)`로 임계값 0.5와 0.7의 예측을 각각 계산하고 출력하세요.
4. Sigmoid(0)을 계산하고, 이 확률을 임계값 0.5와 비교한 예측 클래스도 출력하세요.
5. 확률 표시는 소수점 아래 세 자리로 반올림하되, 임계값 비교에는 반올림 전 p를 사용하세요. Q1~Q5에 답하세요.

#### 작성 도움말

함수의 `return` 앞에는 들여쓰기를 넣습니다. 비교 결과 True와 False를 `.astype(int)`로 바꾸면 1과 0이 됩니다. `>=`에는 기준값과 같은 경우도 포함됩니다.

#### 해석 질문

**Q1.** A의 점수 0.8은 Sigmoid를 거치면 약 얼마의 구매 확률이 되나요? 미구매 확률은 얼마인가요?

**Q2.** B와 C의 구매 확률은 각각 얼마이며, 점수의 크기와 확률의 크기는 어떤 관계인가요?

**Q3.** 임계값 0.5와 0.7에서 예측 클래스는 각각 무엇인가요? 바뀌는 고객은 누구인가요?

**Q4.** 점수 0의 구매 확률은 왜 0.5인가요? 임계값 0.5에서 최종 클래스가 1인 이유는 무엇인가요?

**Q5.** Sigmoid 결과가 0과 1 사이에 있다는 사실만으로 예측 확률이 실제 구매 가능성을 정확히 나타낸다고 보장할 수 있나요?

#### 제출 결과

- Sigmoid 함수, 구매·미구매 확률, 두 임계값의 예측 클래스
- Sigmoid(0)의 계산 과정과 예측 클래스
- Q1~Q5 답변

In [ ]:
# 문제 1-2. Sigmoid 확률과 Threshold로 구매 여부 결정하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.

#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. A의 점수 0.8은 Sigmoid를 거치면 약 얼마의 구매 확률이 되나요? 미구매 확률은 얼마인가요?**

- 답변: 

**Q2. B와 C의 구매 확률은 각각 얼마이며, 점수의 크기와 확률의 크기는 어떤 관계인가요?**

- 답변: 

**Q3. 임계값 0.5와 0.7에서 예측 클래스는 각각 무엇인가요? 바뀌는 고객은 누구인가요?**

- 답변: 

**Q4. 점수 0의 구매 확률은 왜 0.5인가요? 임계값 0.5에서 최종 클래스가 1인 이유는 무엇인가요?**

- 답변: 

**Q5. Sigmoid 결과가 0과 1 사이에 있다는 사실만으로 예측 확률이 실제 구매 가능성을 정확히 나타낸다고 보장할 수 있나요?**

- 답변:

---

## 필수 2. Softmax와 Argmax 이해하기

### 문제 2-1. 한 상품의 Softmax 확률과 클래스 구하기

#### 문제 설명

교안 2·3절의 예제입니다. 한 상품을 의류·식품·가전 중 하나로 분류합니다.

| 클래스 | 의류 | 식품 | 가전 |
|---|---:|---:|---:|
| 인덱스 | 0 | 1 | 2 |
| 점수 z | 2 | 1 | 0 |

Softmax는 **각 점수의 지수값을 구하고 → 그 지수값들을 더하고 → 각 지수값을 합계로 나누는 계산**입니다. 수식에서 Σ(시그마)는 모두 더하라는 뜻입니다.

`Softmax(zᵢ) = e^(zᵢ) / Σ e^(zⱼ)`

i는 지금 확률을 구하려는 클래스의 위치, j는 합계에 넣는 모든 클래스의 위치입니다. 아래 코드는 교안처럼 가장 큰 점수를 먼저 빼고 계산합니다. 모든 점수에서 같은 값을 빼도 Softmax 확률은 유지됩니다.

```python
def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z)
```

함수는 그대로 사용하세요. 이번 문제에서는 이 성질을 증명하지 않고 계산 순서를 따라갑니다.

#### 요구사항

1. `z = np.array([2, 1, 0])`, `classes = np.array(["의류", "식품", "가전"])`를 만드세요.
2. 위 함수를 작성하고 `p = softmax(z)`를 계산하세요.
3. `np.round(p, 3)`과 `np.sum(p)`로 확률과 확률 합을 출력하세요.
4. `index = np.argmax(p)`로 인덱스를 구하고 index와 `classes[index]`를 출력하세요.

#### 작성 도움말

최댓값 2를 각 점수에서 빼면 [0,−1,−2]입니다. 지수값은 약 [1,0.3679,0.1353]이고 합은 약 1.5032입니다. 각 지수값을 이 합으로 나누세요. `np.argmax(p)`는 가장 큰 확률이 있는 위치를 반환합니다.

#### 해석 질문

**Q1.** 최댓값을 뺀 배열과 그 배열의 지수값은 각각 무엇인가요?

**Q2.** 세 클래스의 확률은 각각 얼마이며, 확률 합은 얼마인가요?

**Q3.** Argmax 결과와 예측 클래스 이름은 무엇인가요? Argmax가 확률값을 반환하나요?

**Q4.** 이 문제에서 Softmax를 사용하는 이유는 무엇인가요?

#### 제출 결과

- Softmax 함수와 확률·확률 합
- 예측 인덱스·클래스 이름 및 Q1~Q4 답변

In [ ]:
# 문제 2-1. 한 상품의 Softmax 확률과 클래스 구하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.

#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 최댓값을 뺀 배열과 그 배열의 지수값은 각각 무엇인가요?**

- 답변: 

**Q2. 세 클래스의 확률은 각각 얼마이며, 확률 합은 얼마인가요?**

- 답변: 

**Q3. Argmax 결과와 예측 클래스 이름은 무엇인가요? Argmax가 확률값을 반환하나요?**

- 답변: 

**Q4. 이 문제에서 Softmax를 사용하는 이유는 무엇인가요?**

- 답변:

### 문제 2-2. 여러 고객의 확률과 클래스를 행별로 구하기

#### 문제 설명

교안 2·3절의 고객별 점수 예제입니다. 가로 한 줄인 행은 고객, 세로 한 줄인 열은 클래스입니다.

```python
Z = np.array([[2, 1, 0], [0, 2, 1], [1, 0, 3]])
classes = np.array(["의류", "식품", "가전"])
```

첫 행은 A, 둘째 행은 B, 셋째 행은 C입니다. 클래스 순서는 의류·식품·가전입니다. 한 고객의 세 점수끼리 Softmax를 계산합니다.

아래 교안의 함수를 **그대로 사용**하세요. 여러 행을 계산하는 데 필요한 옵션은 해설에서 다시 풀이합니다.

```python
def softmax(z):
    shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)
```

`axis=1`은 각 행 안의 세 클래스 값을 대상으로 계산하라는 뜻입니다. `keepdims=True`는 고객별로 구한 값이 자신의 행에 적용되도록 결과의 모양을 유지합니다.

#### 요구사항

1. 위의 Z와 classes를 만들고 Softmax 함수를 작성하세요.
2. `p = softmax(Z)`를 계산하고 `np.round(p, 3)`으로 확률을 출력하세요.
3. `Z.shape`, `p.shape`, `np.sum(p, axis=1)`을 출력하세요.
4. `predicted_indices = np.argmax(p, axis=1)`를 계산하고, 인덱스와 `classes[predicted_indices]`를 출력하세요.

#### 작성 도움말

한 행만 먼저 읽어 보세요. 첫 행 [2,1,0]은 앞 문제와 같으므로 같은 확률이 나옵니다. 고객별로 가장 큰 확률이 의류·식품·가전 중 어느 열에 있는지 찾습니다. 함수 안의 axis와 keepdims는 제시된 코드대로 사용하면 됩니다.

#### 해석 질문

**Q1.** Z와 p의 Shape는 무엇이며, 행과 열은 각각 무엇인가요?

**Q2.** 세 고객의 확률 분포와 각 고객의 확률 합은 무엇인가요?

**Q3.** 고객 A·B·C의 예측 인덱스와 클래스 이름은 각각 무엇인가요?

**Q4.** 이 문제에서 axis=1을 지정하는 이유는 무엇인가요?

#### 제출 결과

- 고객별 확률, Z·p의 Shape와 고객별 확률 합
- 고객별 예측 인덱스·이름 및 Q1~Q4 답변

In [ ]:
# 문제 2-2. 여러 고객의 확률과 클래스를 행별로 구하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.

#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. Z와 p의 Shape는 무엇이며, 행과 열은 각각 무엇인가요?**

- 답변: 

**Q2. 세 고객의 확률 분포와 각 고객의 확률 합은 무엇인가요?**

- 답변: 

**Q3. 고객 A·B·C의 예측 인덱스와 클래스 이름은 각각 무엇인가요?**

- 답변: 

**Q4. 이 문제에서 axis=1을 지정하는 이유는 무엇인가요?**

- 답변:

---

## 과제. 필수 실습과 같은 방법으로 복습하기

### 문제 3-1. 같은 확률에 두 임계값 적용하기

#### 문제 설명

교안 연습문제 3의 소문항 1·2를 사용합니다. 이미 구매 확률이 주어져 있으므로 Sigmoid를 다시 적용하지 않습니다.

| 고객 | A | B | C | D |
|---|---:|---:|---:|---:|
| 구매 확률 | 0.85 | 0.40 | 0.65 | 0.20 |

규칙은 **확률이 임계값 이상이면 구매(1), 작으면 미구매(0)**입니다. 같은 확률에 임계값 0.5와 0.7을 각각 적용하세요. 실제 구매 여부는 주어지지 않았습니다.

#### 요구사항

1. `p = np.array([0.85, 0.40, 0.65, 0.20])`를 만드세요.
2. `pred_05 = (p >= 0.5).astype(int)`를 계산하고 출력하세요.
3. `pred_07 = (p >= 0.7).astype(int)`를 계산하고 출력하세요.
4. 답변 작성란에 고객별 두 예측값을 적고, 바뀐 고객을 찾으세요.

#### 작성 도움말

문제 1-2의 비교식을 그대로 사용합니다. 예를 들어 고객 C의 0.65는 0.5 이상이지만 0.7보다 작습니다. 임계값을 바꾸어도 주어진 확률은 그대로입니다.

#### 해석 질문

**Q1.** 임계값 0.5에서 고객 A·B·C·D의 예측은 각각 무엇인가요?

**Q2.** 임계값 0.7에서 고객 A·B·C·D의 예측은 각각 무엇인가요?

**Q3.** 어느 고객의 예측이 바뀌었으며, 그 이유는 무엇인가요?

**Q4.** 임계값을 바꾸면 구매 확률 자체도 바뀌나요? 이 정보만으로 어느 기준이 더 정확한지 알 수 있나요?

#### 제출 결과

- 두 임계값의 예측 배열
- 고객별 비교와 Q1~Q4 답변

In [ ]:
# 문제 3-1. 같은 확률에 두 임계값 적용하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.

#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 임계값 0.5에서 고객 A·B·C·D의 예측은 각각 무엇인가요?**

- 답변: 

**Q2. 임계값 0.7에서 고객 A·B·C·D의 예측은 각각 무엇인가요?**

- 답변: 

**Q3. 어느 고객의 예측이 바뀌었으며, 그 이유는 무엇인가요?**

- 답변: 

**Q4. 임계값을 바꾸면 구매 확률 자체도 바뀌나요? 이 정보만으로 어느 기준이 더 정확한지 알 수 있나요?**

- 답변:

### 문제 3-2. 한 상품의 Softmax와 Argmax 복습하기

#### 문제 설명

교안 연습문제 2의 소문항 3에 있는 점수입니다. 문제 2-1과 같은 방법으로 계산하세요.

| 클래스 | 의류 | 식품 | 가전 |
|---|---:|---:|---:|
| 인덱스 | 0 | 1 | 2 |
| 점수 | 1 | 2 | 1 |

이 한 상품은 세 종류 중 하나로 분류합니다. 교안 3절의 Argmax를 사용해 가장 높은 확률의 클래스도 확인하세요.

#### 요구사항

1. 문제 2-1의 한 벡터용 Softmax 함수를 그대로 작성하세요.
2. `z = np.array([1, 2, 1])`, `classes = np.array(["의류", "식품", "가전"])`를 만드세요.
3. `p = softmax(z)`를 계산하고 `np.round(p, 3)`과 `np.sum(p)`를 출력하세요.
4. `index = np.argmax(p)`를 계산하고 index와 `classes[index]`를 출력하세요.

#### 작성 도움말

한 벡터용 함수에는 axis=1을 쓰지 않습니다. 가장 큰 점수 2를 빼면 [−1,0,−1]이고, 지수값은 약 [0.3679,1,0.3679]입니다. 이 세 값을 더한 뒤 각각을 같은 합으로 나누세요.

#### 해석 질문

**Q1.** 최댓값을 뺀 배열, 지수값, 지수값의 합은 각각 무엇인가요?

**Q2.** 의류·식품·가전의 확률과 확률 합은 각각 얼마인가요?

**Q3.** Argmax 결과와 최종 예측 클래스는 무엇인가요? 인덱스 1은 확률 100%라는 뜻인가요?

**Q4.** 의류와 가전의 확률이 같은 이유는 무엇인가요?

#### 제출 결과

- 한 벡터용 Softmax 함수, 확률·확률 합
- 예측 인덱스·클래스 이름과 Q1~Q4 답변

In [ ]:
# 문제 3-2. 한 상품의 Softmax와 Argmax 복습하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.

#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 최댓값을 뺀 배열, 지수값, 지수값의 합은 각각 무엇인가요?**

- 답변: 

**Q2. 의류·식품·가전의 확률과 확률 합은 각각 얼마인가요?**

- 답변: 

**Q3. Argmax 결과와 최종 예측 클래스는 무엇인가요? 인덱스 1은 확률 100%라는 뜻인가요?**

- 답변: 

**Q4. 의류와 가전의 확률이 같은 이유는 무엇인가요?**

- 답변:

---

## 실습 마무리

1. 회귀 예측값, 분류 점수, 예측 확률, 예측 클래스는 어떻게 다른가요?
2. Sigmoid(0)은 얼마이며, 임계값 0.5에서는 어떤 클래스로 정하나요?
3. 구매 확률이 p일 때 미구매 확률은 무엇인가요?
4. Softmax는 무엇을 더한 합으로 각 지수값을 나누나요?
5. 여러 고객의 확률에서 Argmax의 axis=1은 무엇을 뜻하나요?
6. 임계값을 높이거나 가장 높은 확률을 고르면 반드시 더 정확한 예측인가요?

### 실습 마무리 답변 작성란

**1. 회귀 예측값, 분류 점수, 예측 확률, 예측 클래스는 어떻게 다른가요?**

- 답변: 

**2. Sigmoid(0)은 얼마이며, 임계값 0.5에서는 어떤 클래스로 정하나요?**

- 답변: 

**3. 구매 확률이 p일 때 미구매 확률은 무엇인가요?**

- 답변: 

**4. Softmax는 무엇을 더한 합으로 각 지수값을 나누나요?**

- 답변: 

**5. 여러 고객의 확률에서 Argmax의 axis=1은 무엇을 뜻하나요?**

- 답변: 

**6. 임계값을 높이거나 가장 높은 확률을 고르면 반드시 더 정확한 예측인가요?**

- 답변: